### Mounting google drive

In [85]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Importing the required libaries

In [86]:
import pandas as pd
import spacy
import re

### Quick exploration

In [87]:
df = pd.read_csv("/content/drive/MyDrive/cleaned/cleaned_reviews.csv")

In [88]:
df.head(10)

,product_id,reviewer_rating,review_text
0,381863621,1,Over size. I ordered for medium boxer and what...
1,381863621,3,Allow colours to be choosen. My hubby loves it...
2,381863621,5,I like it. It looks so good
3,381863621,1,I don’t like it. This not the item I purchased...
4,381863621,5,It's good product. It's good product for those...
5,381863621,5,I like it. Very good condom boxers... I really...
6,381863621,5,Stretchy. It’s okay. I recommend
7,381863621,5,nice condom boxer. Nice
8,381863621,5,i love the boxers about how they did it. It ve...
9,381863621,4,Good. Good product


In [89]:
df.loc[0, 'review_text']

'Over size. I ordered for medium boxer and what I got was over size boxer which I couldn’t wear'

### Deduplicating reviews

In [90]:
def deduplicate_within_review(review):
    sents = [s.strip() for s in re.split(r'[.!?]', review) if s.strip()]

    seen = set()
    unique_sentences = []

    for sen in sents:
        cleaned = sen.lower()

        is_duplicate = False

        for seen_s in seen:
            if cleaned in seen_s or seen_s in cleaned:
                is_duplicate = True
                if len(cleaned) > len(seen_s):
                    seen.remove(seen_s)
                    seen.add(cleaned)
                    unique_sentences = [cleaned if s.lower() == seen_s else s for s in unique_sentences]
                break

        if not is_duplicate:
            seen.add(cleaned)
            unique_sentences.append(sen)

    return '. '.join(unique_sentences)

In [91]:
print(f"Original_review: {df.loc[0, 'review_text']}")
print(f"Deduplicated_review: {deduplicate_within_review(df.loc[0, 'review_text'])}")

Original_review: Over size. I ordered for medium boxer and what I got was over size boxer which I couldn’t wear
Deduplicated_review: i ordered for medium boxer and what i got was over size boxer which i couldn’t wear


In [92]:
print(f"Original_review: {df.loc[10, 'review_text']}")
print(f"Deduplicated_review: {deduplicate_within_review(df.loc[10, 'review_text'])}")

Original_review: improve on the texture and fabric. improve on the texture and fabric
Deduplicated_review: improve on the texture and fabric


In [93]:
print(f"Original_review: {df.loc[20, 'review_text']}")
print(f"Deduplicated_review: {deduplicate_within_review(df.loc[20, 'review_text'])}")

Original_review: my son love it. lovely
Deduplicated_review: my son love it. lovely


In [94]:
print(f"Original_review: {df.loc[500, 'review_text']}")
print(f"Deduplicated_review: {deduplicate_within_review(df.loc[500, 'review_text'])}")

Original_review: What is ordered.. The bracelets, naah. The watch is good.
But. Exactly what is ordered.
Deduplicated_review: exactly what is ordered. The bracelets, naah. The watch is good. But


In [95]:
df['deduplicated_review'] = df['review_text'].apply(deduplicate_within_review)

### Experimenting with groq

In [96]:
!pip install groq

In [97]:
from groq import Groq

In [98]:
api_key = "your_api_key"

In [99]:
prompt = """
You are an expert at analyzing product reviews for fashion items (shoes, boxers, clothes, wristbands, etc.).

Your task is to extract **aspects** and their **sentiment** from a given review.

Use the following **aspect categories only**:
- sizing
- quality
- design
- item_accuracy
- delivery
- color_options
- comfort

For each review:
- Identify which aspects are mentioned.
- For each aspect mentioned, classify the sentiment as: "positive", "negative", or "neutral".
- If an aspect is not mentioned, do not include it.
- Return the result strictly as **JSON**, in the following format:

{{
  "sizing": "positive/negative/neutral",
  "quality": "positive/negative/neutral",
  "design": "positive/negative/neutral",
  "item_accuracy": "positive/negative/neutral",
  "delivery": "positive/negative/neutral",
  "color_options": "positive/negative/neutral",
  "comfort": "positive/negative/neutral"
}}

Example:

Review: "The shoes fit perfectly, the color was exactly as advertised, but the material feels cheap."

Output:
{{
  "sizing": "positive",
  "quality": "negative",
  "item_accuracy": "positive"
}}

Now analyze the following review and return only the JSON:
"{review_text}"
"""

In [100]:
client = Groq(api_key=api_key)

def extract_aspect_and_sentiment_from_review(review_text):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "user", "content": prompt.format(review_text=review_text)}
            ]
        )

        result = response.choices[0].message.content
        return result

    except Exception as e:
        print("Error:", e)
        return None

In [103]:
ten_reviews = df['deduplicated_review'].iloc[10:20].tolist()

In [104]:
for review in ten_reviews:
  print(review)
  print(extract_aspect_and_sentiment_from_review(review))
  print()

improve on the texture and fabric
{
  "quality": "negative"
}

No under lining. This pant is without and under lining, just lace all true. You have to better by adding a lining to the under so when u get wet or stained, it will hold
{
  "design": "negative",
  "quality": "negative"
}

wonderful. Exactly as seen so soft and smooth
{
  "item_accuracy": "positive",
  "quality": "positive",
  "comfort": "positive"
}

Excellent. Exactly what I ordered 
I love it 
Definitely getting Another one
{
  "item_accuracy": "positive",
  "quality": "positive"
}

nice. Satisfactory
{
  "quality": "positive"
}

It does not hv cotton at the vj area. I don’t like it , I had to give it out
{
  "quality": "negative",
  "item_accuracy": "negative"
}

Oh my my. It suits my wife so well. I love them on her. A goo buy
{
  "sizing": "positive",
  "item_accuracy": "positive"
}

it was fine. Perfect
{
  "quality": "positive"
}

Ok. Lovely pants
{
  "design": "positive"
}

Sexy. My wife love it
{
  "design": "posi

### Saving the dataset

In [105]:
from google.colab import files

In [106]:
df.to_csv('deduplicated_reviews.csv', index=False)

In [107]:
files.download('deduplicated_reviews.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>